<a href="https://colab.research.google.com/github/Yunajinn/MJO_activity/blob/main/phase_bargraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Set up ipython environment
# You don't need to modify anything here.
import gdown
import os
import json
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

stndata_loc = "/content/PAC/"
MJOind_loc = "/content/MJO_indices/"
PICASO_loc = "/content/"
download_loc = PICASO_loc + "Download/"
os.makedirs(download_loc, exist_ok = True)
os.makedirs(download_loc, exist_ok=True)
os.makedirs(stndata_loc + "data_country/", exist_ok=True)
os.makedirs(MJOind_loc, exist_ok=True)

# Download files
json_file_id = '1kcIbLDxD-hF0VldFFpl1ydfKNM7GGybI'
if not os.path.exists('file_ids.json') :
  print("Downloading configuration file (file_ids.json)...")
  gdown.download(id=json_file_id, output='file_ids.json', quiet=True, fuzzy=True, use_cookies=False)


# Download essential common files
if (not os.path.exists("/content/PICASO_functions.py")
    or not os.path.exists("rmm.74to25.nc")
    or not os.path.exists("stn_lon360.txt")) :
  with open('file_ids.json', 'r') as f:
      file_ids = json.load(f)
  print("\nDownloading essential common files...")
  gdown.download(id=file_ids['common']['PICASO_functions'],
                output="/content/PICASO_functions.py", quiet=True, fuzzy=True, use_cookies=False) if not os.path.exists("/content/PICASO_functions.py") else None
  gdown.download(id=file_ids['common']['rmm_index'],
                output=f"rmm.74to25.nc", quiet=True, fuzzy=True, use_cookies=False) if not os.path.exists("rmm.74to25.nc") else None
  gdown.download(id=file_ids['common']['station_loc'],
                output=f"stn_lon360.txt", quiet=True, fuzzy=True, use_cookies=False) if not os.path.exists("stn_lon360.txt") else None

import PICASO_functions as fn


# Part 0: Prepare the analysis
### Select your country and choose one station.   

Country name list (in alphabetical order):  
    `Cook_Islands`, `F.S.Micronesia`, `Fiji`, `Kiribati`, `Marshall_Islands`, `Nauru`, `New_Caledonia`, `Niue`, `Palau`, `Papua_New_Guinea`, `Samoa`, `Solomon_Islands`, `Tonga`, `Tuvalu`, `Vanuatu`


Insert your country in `my_country` in the format of country name list.       
Insert the name of selected station in `selected_station`.      
Select the season for each variables.  
Put in the starting month in `month_strt`, and ending month in `month_end`.

In [ ]:
my_country = 'Fiji'

In [ ]:
stations = fn.station_loc_sel(stndata_loc=PICASO_loc, country = my_country)
print(f"Station names in {my_country}:")
print('   ', "\n    ".join(stations.name.dropna().unique()))

In [ ]:
selected_station = 'Nadi Airport'
month_strt = 11
month_end = 3

In [ ]:
def download_load_data(my_country, selected_station) :
    stations = fn.station_loc_sel(stndata_loc=PICASO_loc, country = my_country)
    if selected_station not in stations.name.dropna().unique() :
        raise SystemError(f"Selected station is not in the list.")

    stn_lat = stations[stations["name"] == selected_station].lat.item()
    stn_lon = stations[stations["name"] == selected_station].lon.item()

    print(f"\nDownloading and loading data for {my_country}...")

    # Load configuration file
    with open('file_ids.json', 'r') as f:
        file_ids = json.load(f)
    ids = file_ids['countries'][my_country]
    stndata_loc = "/content/PAC/"
    imerg_path = f"{stndata_loc}data_country/IMERG_PAC.98to25.{my_country}.nc"
    era5_path = f"{stndata_loc}data_country/ERA5_t2m_PAC.98to25.{my_country}.nc"

    # Download reanalysis data for my country
    if not os.path.exists(imerg_path) :
        print(f"\nDownloading and loading precipitation data for {my_country}...")
        gdown.download(id=ids['IMERG'], output=imerg_path, quiet=False, fuzzy=True, use_cookies=False)

    if not os.path.exists(era5_path) :
        print(f"\nDownloading and loading 2m temperature data for {my_country}...")
        gdown.download(id=ids['ERA5'], output=era5_path, quiet=False, fuzzy=True, use_cookies=False)

    # Load data into memory
    # Select one nearest grid from the selected station in the reanalysis data
    precip = xr.open_dataset(imerg_path).precipitation.sel(lat = stn_lat, lon = stn_lon, method = 'nearest')
    t2m = xr.open_dataset(era5_path).t2m.sel(lat = stn_lat, lon = stn_lon, method = 'nearest')

    t2m = t2m - 273.15 # reset temperature unit from K to °C

    # See how the original data look like
    fig, ax = plt.subplots(1, 2, figsize = (16, 5))
    plt.suptitle(f"Original data nearest from {selected_station}, {my_country}",  fontsize = 20)
    ax[0].set_title("IMERG precipitation data", loc = 'left', fontsize = 13)
    ax[0].plot(precip.time, precip, color = 'tab:blue', linewidth = 0.8)
    ax[0].set_ylabel('mm/day', fontsize = 12)
    ax[0].set_xlabel('time', fontsize = 12)

    ax[0].set_ylim(0, 280)
    ax[0].set_yticks(np.arange(0, 251, 50))
    ax[0].set_xlim(np.datetime64('1998-01-01'), np.datetime64('2026-01-01'))
    # ax[0].set_xticks(np.arange(1998, 2025, 3))

    ax[1].set_title("ERA5 2m temperature data", loc = 'left', fontsize = 13)
    ax[1].plot(t2m.time, t2m, color = 'tab:red', linewidth = 0.8)
    ax[1].set_ylabel('°C', fontsize = 12)
    ax[1].set_xlabel('time', fontsize = 12)
    ax[1].set_ylim(17, 30)
    ax[1].set_yticks(np.arange(18, 31, 2))
    ax[1].set_xlim(np.datetime64('1998-01-01'), np.datetime64('2026-01-01'))
    # ax[1].set_xticks(np.arange(1998, 2025, 3))

    plt.tight_layout()
    plt.savefig(download_loc + f'{my_country}.{selected_station}.original_data.png',  bbox_inches = 'tight', dpi = 100)
    plt.show()
    plt.close()

    return precip, t2m

precip, t2m = download_load_data(my_country, selected_station)

## We calculate the seasonal cycle (climatology).

In [ ]:
def calculate_climatology(data_array: xr.DataArray, vname: str, harmonic_order=3, save_fname=None):
    """
    Calculate the climatology using Harmonic Analysis.
    This method first computes the annual cycle from daily data,
    then reconstructs a smooth seasonal cycle using the 1st to 3rd harmonics.
    This keeps only the large-scale seasonal variations to produce a smoother climatological reference.
    """

    # 1. Create a 'date' coordinate from the time dimension using MM-DD format.
    if "date" not in data_array.coords:
        raw_data = data_array.assign_coords(
            date=("time", data_array.time.dt.strftime("%m-%d").values)
        )
    else:
        raw_data = data_array

    # Check whether February 29 exists in the dataset.
    leap_yr_opt = (raw_data.date == "02-29").any().item()

    # 2. Compute the basic statistics of the annual cycle. We estimate the average value for each calendar day.
    if leap_yr_opt:
        # Extrack Februray 28 and March 1 from non-leap years, then use their average to estimate February 29.
        # This is done so that leap-day climatology is not based only on leap years.
        non_leap_feb_28 = raw_data.sel(time=~raw_data.time.dt.is_leap_year & (raw_data.date == "02-28"))
        non_leap_mar_01 = raw_data.sel(time=~raw_data.time.dt.is_leap_year & (raw_data.date == "03-01"))

        if (non_leap_feb_28.time.size == non_leap_mar_01.time.size and non_leap_feb_28.time.size > 0 ):
            non_leap_feb_29 = (non_leap_feb_28.drop_vars(["time", "date"]) + non_leap_mar_01.drop_vars(["time", "date"])) / 2.0
            non_leap_feb_29 = non_leap_feb_29.assign_coords({"time": [pd.Timestamp("2000-02-29")] * non_leap_feb_29.sizes["time"],
                                                            "date": "02-29"})
            concat_data = xr.concat([raw_data, non_leap_feb_29], dim="time")
            annual_cyc = concat_data.groupby("date").mean(dim="time").sortby("date")
        else:
            annual_cyc = raw_data.groupby("date").mean(dim="time").sortby("date")
    else:
        annual_cyc = raw_data.groupby("date").mean(dim="time").sortby("date")

    # 3. Prepare harmonic analysis.
    D = annual_cyc["date"].size
    t = np.arange(1, D + 1)

    # Generate cosine and sine basis functions for each harmonic.
    # These basis functions represent repeating seasonal signals.
    cos_term = xr.DataArray(
        np.array([np.cos(2 * i * np.pi * t / D) for i in range(1, harmonic_order + 1)]),
        dims=["n", "date"], coords={"n": np.arange(1, harmonic_order + 1), "date": annual_cyc["date"]} )
    sin_term = xr.DataArray(np.array([np.sin(2 * i * np.pi * t / D) for i in range(1, harmonic_order + 1)]),
        dims=["n", "date"], coords={"n": np.arange(1, harmonic_order + 1), "date": annual_cyc["date"]})

    # Compute harmonic coefficients A and B.
    A = (2 / D) * (annual_cyc * cos_term).sum(dim="date")
    B = (2 / D) * (annual_cyc * sin_term).sum(dim="date")
    harmonics = (A * cos_term + B * sin_term).sum(dim="n")

    # Compute the mean over the full annual cycle.
    # This is the background mean level around which the seasonal cycle varies.
    time_mean = annual_cyc.mean(dim="date")

    # 4. Construct the final climatology as:
    #    climatology = annual mean + reconstructed harmonic seasonal cycle
    dim_order = ["date"] + [dim for dim in data_array.dims if dim != "time"]
    climatology = (harmonics + time_mean).transpose(*dim_order)

    # See how climatology looks like
    fig, ax = plt.subplots(figsize = (8, 4))

    if vname == 'precip' :
      text = 'IMERG precipitation'
      unit = 'mm/day'
      ylim = (0, 21, 2)
      color = 'tab:blue'

    elif vname == 't2m' :
      text = 'ERA5 t2m'
      unit = '°C'
      ylim = (18, 31, 2)
      color = 'tab:red'

    plt.suptitle(f"Climatology of {selected_station}, {my_country}",  fontsize = 16)
    ax.annotate(text, xy=(0.98, 0.98), xycoords='axes fraction', ha='right', va='top', fontsize=12)
    ax.plot(climatology.date, climatology, color = color)
    ax.set_ylabel(unit, fontsize = 12)
    ax.set_xlabel('date', fontsize = 12)
    ax.set_ylim(ylim[0], ylim[1])
    ax.set_yticks(np.arange(ylim[0], ylim[1], ylim[2]))
    ax.set_xlim(climatology.date.min(), climatology.date.max())
    ax.set_xticks(climatology.date[::30])

    plt.savefig(download_loc + f'{my_country}.{selected_station}.{vname}.climatology.png',  bbox_inches = 'tight', dpi = 100)
    plt.show()
    plt.close()
    print()
    print()

    return climatology

precip_climatology = calculate_climatology(precip, vname = 'precip', harmonic_order= 3, save_fname = None)
t2m_climatology = calculate_climatology(t2m, vname = 't2m', harmonic_order = 3, save_fname = None)

## We calculate anomaly by subtracting climatology over original data

In [ ]:
def calc_anomaly(var, clim, varname) :
    # Match each original time step to the corresponding climatological calendar day.
    date_key = xr.DataArray(var.time.dt.strftime('%m-%d').values, coords={'time': var.time}, dims='time')
    var_anom = var - clim.sel(date=date_key)

    # Remove the linear trend from the anomaly time series.
    # This step keeps short-term variability while excluding long-term drift.
    fit = var_anom.polyfit(dim='time', deg=1)
    trend = xr.polyval(var_anom['time'], fit.polyfit_coefficients)
    var_anom = var_anom - trend
    var_anom.name = varname

    if varname == 'precip' :
      long_name = 'Precipitation'
      unit = 'mm/day'
      color = 'tab:blue'
      text = 'IMERG precipitation'
      ylim = (-50, 251, 50)

    elif varname == 't2m' :
      long_name = '2m temperature'
      unit = '°C'
      color = 'tab:red'
      text = 'ERA5 t2m'
      ylim = (-5, 5.1, 1)

    # See how the anomaly data look like
    fig, ax = plt.subplots(figsize = (8, 5))
    plt.suptitle(f"{long_name} anomaly of {selected_station}, {my_country}",  fontsize = 18)
    ax.plot(var_anom.time, var_anom, color = color, linewidth = 0.8)
    ax.annotate(text, xy=(0.98, 0.98), xycoords='axes fraction', ha='right', va='top', fontsize=12)

    ax.set_ylabel(unit, fontsize = 12)
    ax.set_xlabel('time', fontsize = 12)
    ax.set_ylim(ylim[0], ylim[1])
    ax.set_yticks(np.arange(ylim[0], ylim[1], ylim[2]))
    ax.set_xlim(np.datetime64('1998-01-01'), np.datetime64('2026-01-01'))

    plt.tight_layout()
    plt.savefig(download_loc + f'{my_country}.{selected_station}.{varname}.anomaly.png',  bbox_inches = 'tight', dpi = 100)
    plt.show()
    plt.close()
    print()
    print()
    return var_anom

precip_anomaly = calc_anomaly(precip, precip_climatology, varname = 'precip')
t2m_anomaly = calc_anomaly(t2m, t2m_climatology, varname = 't2m')


Select season from `month_strt` to `month_end` we selected before.  

In [ ]:
def year_month_select(year, month_strt, month_end) :
    if month_strt > month_end :
        timestrt = f"{year}-{month_strt:02d}-01"
        last_day = (pd.Timestamp(year=year+1, month=month_end, day=1) + pd.offsets.MonthEnd(0)).day
        timelast = f"{year+1}-{month_end:02d}-{last_day}"
    else :
        timestrt = f"{year}-{month_strt:02d}-01"
        last_day = (pd.Timestamp(year=year, month=month_end, day=1) + pd.offsets.MonthEnd(0)).day
        timelast = f"{year}-{month_end:02d}-{last_day}"
    return timestrt, timelast

def season_sel(ds, month_strt, month_end) :
    month_list = []
    if month_strt > month_end :
        for month in range(month_strt, 13) :
            month_list.append(month)
        for month in range(1, month_end + 1) :
            month_list.append(month)
    else :
        for month in range(month_strt, month_end + 1) :
            month_list.append(month)

    return ds.where(ds.time.dt.month.isin(month_list), drop = True)

def month_select(var_anomaly, month_strt, month_end, year = None) :
    if year is not None :
        if not isinstance(year, int):
            raise ValueError(f"year must be int or None, got {year!r}")

        timestrt, timelast = year_month_select(year, month_strt = month_strt, month_end = month_end)
        var_anomaly = var_anomaly.sel(time = slice(timestrt, timelast))

    else:
        var_anomaly = season_sel(var_anomaly, month_strt = month_strt, month_end = month_end)

    return var_anomaly

precip_anomaly = month_select(precip_anomaly, month_strt = month_strt, month_end = month_end, year = None)
t2m_anomaly = month_select(t2m_anomaly, month_strt = month_strt, month_end = month_end, year = None)

# Section 1. Reading the graphs   

First, we investigate which information contains in MJO indices (`phase`, `amplitude`).  
We use the MJO amplitude to determine whether the MJO was active or not.      
When the MJO was active, we use the MJO phase to identify the location of dry and wet anomalies in the tropics.  
Then, we seperate the data for each variable (`t2m_anomaly`, `precip_anomaly`) according to MJO phase and amplitude.  
As a result, we obtain anomalies for nine MJO phases with dimensons of `(time, MJO phase)`.    
Note that we selected NDJFM season from above.


In [ ]:
# import MJO indices
MJO_indices = xr.open_dataset("/content/" + "rmm.74to25.nc")
MJO_amplitude, MJO_phase = MJO_indices.amplitude, MJO_indices.phase


In [ ]:
def draw_phase_diagram(MJO_indices, starting_time = '2020-01-01', ending_time = '2020-02-29', vmax=5, radius=1):

  timerange = slice(starting_time, ending_time)
  RMM1 = MJO_indices.RMM1.sel(time = timerange)
  RMM2 = MJO_indices.RMM2.sel(time = timerange)

  fig, ax = plt.subplots(figsize= (6, 6))

  # divide phases
  for i in range(2):
      ax.plot([-vmax*0.85*np.cos(i*np.pi/2),  vmax*0.85*np.cos(i*np.pi/2)],
              [-vmax*0.85*np.sin(i*np.pi/2),  vmax*0.85*np.sin(i*np.pi/2)],
              ls='--', c='gray', lw=0.8)
      ax.plot([-vmax*np.sqrt(2)*np.cos((2*i+1)*np.pi/4), vmax*np.sqrt(2)*np.cos((2*i+1)*np.pi/4)],
              [-vmax*np.sqrt(2)*np.sin((2*i+1)*np.pi/4), vmax*np.sqrt(2)*np.sin((2*i+1)*np.pi/4)],
              ls='--', c='gray', lw=0.8)

  # draw circle (amplitude)
  ax.add_patch(plt.Circle((0, 0), radius, facecolor='white', zorder=2))
  ax.add_patch(plt.Circle((0, 0), radius, edgecolor='k', facecolor='none', lw=1, zorder=3))

  zones = ["Maritime\nContinent", "Western\nPacific",
          "West. Hem.\nand Africa", "Indian\nOcean"]
  for i, zone in enumerate(zones):
      x = vmax*0.92*np.cos(i*np.pi/2)
      y = vmax*0.92*np.sin(i*np.pi/2)
      rot = 90*((i+1)%2)*(-1)**(i//2+1)
      ax.text(x, y, zone, ha='center', va='center', fontsize=11, rotation=rot)

  for p in range(1, 9):
      ang = p*np.pi/4 - np.pi/8
      ax.text(-vmax*np.cos(ang), -vmax*np.sin(ang),
              str(p), ha='center', va='center',
              fontsize=14, weight='bold')


  ax.set_xlim(-vmax, vmax)
  ax.set_ylim(-vmax, vmax)
  ax.set_xlabel('RMM1', fontsize = 14)
  ax.set_ylabel('RMM2', fontsize = 14)
  ax.set_aspect('equal')
  ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.3)
  ax.tick_params(labelsize=10)

  ax.plot(RMM1, RMM2, marker = '.', color='black', alpha = 0.6,  label='RMM')
  ax.scatter(RMM1[0], RMM2[0], color='red', zorder=10, label = 'Start')
  ax.scatter(RMM1[-1], RMM2[-1], color='blue', zorder=11, label = 'End')
  ax.legend(loc = 'upper left', handlelength = 0.7)
  plt.savefig(download_loc + f'MJO_phase_digram.{starting_time}-{ending_time}.png',  bbox_inches = 'tight', dpi = 100)
  plt.show()
  plt.close()

draw_phase_diagram(MJO_indices, starting_time = '2020-01-01', ending_time = '2020-02-29')

In [ ]:
# Set MJO phases based on the MJO amplitude.
# If the amplitude is below 1.0, set the phase to 0.
phase_label = xr.where(MJO_amplitude >= 1.0, MJO_phase, 0).rename("phase")

# Assign the MJO phase label to each day and use it as a coordinate for each variable.
precip_phase = precip_anomaly.assign_coords(phase = phase_label.sel(time=precip_anomaly.time))
t2m_phase = t2m_anomaly.assign_coords(phase = phase_label.sel(time=t2m_anomaly.time))

In [ ]:
def plot_bar_graph(var_phase, country, station, sig_buffer = None, colors = ['tab:blue', 'tab:red'], varmax = 5, vartick = 1) :
    name = var_phase.name
    if name == 't2m' :
        units = '2m temperature anomaly (K)'
        text = 'ERA5 daily 2m temperature'
    elif name == 'precip':
        units = 'Precipitation anomaly (mm/d)'
        text = 'IMERG daily precipitation'

    download_loc = "/content/Download/"
    phase_list = range(9)
    yticks = np.arange(-varmax, varmax + 0.1, vartick)
    values = [var_phase.where(var_phase.phase == ph, drop=True).mean(dim = 'time').item() for ph in phase_list]
    bar_colors = [colors[1] if val >= 0 else colors[0] for val in values]

    plt.close('all')
    fig, ax = plt.subplots(figsize = (5, 5))

    ax.axhline(0, color ='black', linewidth=1.01, linestyle='-', zorder = 25)
    ax.axvline(0.5, color = 'black', linewidth = 1., linestyle = '--', alpha = 0.8, zorder = -1)
    ax.bar(np.arange(0, 9), values, bottom=0, color=bar_colors, width= 0.65, zorder = 11)
    ax.set_xlabel('MJO phase', fontsize = 15)
    ax.set_title(f'{country}: {station}', fontsize = 17, loc = 'left', weight = 'semibold')
    ax.set_yticks(yticks)
    ax.set_ylim(min(yticks), max(yticks))
    ax.set_xticks(ticks=np.arange(len(phase_list)), labels=phase_list, fontsize = 12)
    ax.set_ylabel(units, fontsize = 15)
    ax.tick_params(axis='y', labelsize = 12)
    ax.annotate(text, xy=(0.98, 0.98), xycoords='axes fraction', ha='right', va='top', fontsize=12)


    plt.tight_layout()
    plt.savefig(download_loc + f'{name}.{country}.{station}.MJO_phase.png',  bbox_inches = 'tight', dpi = 100)
    plt.show()
    plt.close()
    print()
    print()

plot_bar_graph(t2m_phase, country = my_country, station = selected_station, varmax = 0.2, vartick = 0.05)
plot_bar_graph(precip_phase, country = my_country, station = selected_station, varmax = 10, vartick = 2)



# Section 2. Phase frequency and interpretation   

In this section, we examine the frequency of each MJO phase.  
Earlier, we discussed how we define MJO amplitude and phase by `RMM1` and `RMM2`, then used `MJO_phase` and `MJO_amplitude` to seperate each variable by MJO phase.   
Here, instead of applying the indices directly to the atmospheric variables, we use the MJO indices themselves to investigate phase frequency in selected years.   
Note that, in the previous analysis, `t2m` and `precipitation` were restricted to the selected season.   
Year 1980 to 2025 is provided here.

In [ ]:
def count_phase_time(MJO_indices, year, month_strt = 1, month_end = 12, amplitude_threshold = 1.0) :
    MJO_amplitude, MJO_phase = MJO_indices.amplitude, MJO_indices.phase

    timestrt, timelast = year_month_select(year, month_strt, month_end)
    MJO_phase_year = MJO_phase.sel(time = slice(timestrt, timelast))
    MJO_amplitude_year = MJO_amplitude.sel(time = slice(timestrt, timelast))
    phase_chosen = xr.where(MJO_amplitude_year >= amplitude_threshold, MJO_phase_year, 0)

    counts = []
    for phase in range(9):
        n = (phase_chosen == phase).sum().item()
        counts.append(n)

    phase_table = pd.DataFrame({"count (days)":counts}, index=pd.Index(range(9), name = "MJO phase"))
    print(f"Counted days for each MJO phase in year {year}:")
    print()
    print(phase_table)

In [ ]:
year = 2020
count_phase_time(MJO_indices, year = year)